# Ejercicio: rescate en un parque nacional

Un excursionista se ha perdido en un parque nacional y un equipo de rescate debe llegar hasta su ubicación de la forma más eficiente posible. El parque está dividido en varios puntos de referencia y cada camino tiene un coste asociado, determinado por factores como la distancia, el terreno y el clima.

## Objetivo
Encontrar el camino de menor coste entre el punto de partida del equipo de rescate y la ubicación del excursionista.

## Requisitos
1. **Definir el grafo**: cada punto de referencia es un nodo y cada camino entre dos puntos es una arista con un coste asociado.
2. **Implementar una búsqueda de coste uniforme o informada** (por ejemplo, A*) para encontrar el camino óptimo.
3. Comparar los resultados de distintos algoritmos, como la búsqueda en anchura, la búsqueda de coste uniforme y A*.

## Descripción del grafo
Para simplificar, representaremos los nodos con letras (por ejemplo, «A», «B», «C») y asignaremos un coste a cada camino. A continuación se muestra el grafo inicial en Python.

In [2]:
# Definició del graf amb costos d'acció
# Cada clau representa un node i el seu valor és un diccionari amb els nodes veïns i el cost per arribar-hi

graf = {
    "A": {"B": 4, "C": 3},
    "B": {"A": 4, "D": 5, "E": 12},
    "C": {"A": 3, "F": 7},
    "D": {"B": 5, "E": 4, "G": 10},
    "E": {"B": 12, "D": 4, "G": 2, "H": 5},
    "F": {"C": 7, "H": 6},
    "G": {"D": 10, "E": 2, "I": 1},
    "H": {"E": 5, "F": 6, "I": 3},
    "I": {"G": 1, "H": 3}
}

In [16]:
import queue

def ucs(graf, inici, final):
    visitats = set()
    frontera = queue.PriorityQueue()
    frontera.put((0, inici, [inici]))

    while not frontera.empty():
        cost_actual, node, cami = frontera.get()
        print(cami)
        
        visitats.add(node)

        if node == final:
            return cost_actual, final, cami
        
        for desti, cost in graf[node].items():
            if desti not in visitats:
                frontera.put((cost_actual + cost, desti, cami + [desti]))

    return None

cost, desti, cami = ucs(graf, "A", "I")
cami

['A']
['A', 'C']
['A', 'B']
['A', 'B', 'D']
['A', 'C', 'F']
['A', 'B', 'D', 'E']
['A', 'B', 'D', 'E', 'G']
['A', 'B', 'E']
['A', 'C', 'F', 'H']
['A', 'B', 'D', 'E', 'G', 'I']


['A', 'B', 'D', 'E', 'G', 'I']

In [ ]:
from dataclasses import dataclass

@dataclass
class Estat:
    node: str = ""
    pare: 'Estat' = None

    def __lt__(self, other):
        return self.node < other.node
        

def ucs_estats(graf, inici, final):
    estat_inicial = Estat(inici, None)
    visitats = set()
    frontera = queue.PriorityQueue()

    frontera.put((0, estat_inicial))

    while not frontera.empty():
        cost_actual, estat = frontera.get()
        
        node = estat.node
        visitats.add(estat.node)

        if node == final:
            return cost_actual, estat

        for desti, cost in graf[node].items():
            if desti not in visitats:
                nou_estat = Estat(desti, estat)

                frontera.put((cost_actual + cost, nou_estat))

    return None

cost_actual, estat = ucs_estats(graf, "A", "I")

while estat.pare is not None:
    print(estat.node)
    estat = estat.pare


I
G
E
D
B


## Tareas

- **Implementa el algoritmo de búsqueda** que consideres adecuado para encontrar el camino de menor coste desde el inicio hasta el destino.
- **Calcula y muestra el camino y el coste total** del recorrido encontrado.
- **Analiza los resultados**: compara la ruta obtenida con distintos métodos de búsqueda si pruebas más de uno (por ejemplo, búsqueda de coste uniforme y A*).

### Punto de partida y destino

- **Punto de inicio**: nodo «A».
- **Punto de destino** (ubicación del excursionista): nodo «I».

## Solución

Primero implementaremos UCS (*Uniform Cost Search*) para encontrar el camino de menor coste entre el punto inicial y el destino. Por ahora no reconstruiremos la ruta; nos centraremos en encontrar el coste total.

In [9]:
import queue

def ucs(graf, node_inicial, node_final):
    # Inicialitzem la llista de nodes visitats
    visitats = set()
    # Inicialitzem la llista de nodes per visitar
    frontera = queue.PriorityQueue()
    # Inicialitzem la frontera amb el node inicial
    frontera.put((0, node_inicial))
    # Per poder comparar els estats avaluats incialitzem avaluats a 0
    avaluats = 0

    # Mentre la frontera no estigui buida
    while not frontera.empty():
        # Treiem el node de la frontera
        (cost, node) = frontera.get()
        # Incrementem el nombre d'estats avaluats
        avaluats += 1

        # Si el node no ha estat visitat
        if node[-1] not in visitats:
            # Marquem el node com a visitat
            visitats.add(node)

            # Si el node és el node final, retornem el camí
            if node == node_final:
                return node, cost, avaluats

            # Afegim els veïns del node a la frontera, cost = ucs(graf, node_inicial, node_final)
            for vei, cost_vei in graf[node[-1]].items():
                if vei not in visitats:
                    frontera.put((cost + cost_vei, vei))

    # Si no hi ha camí, retornem None
    return None

# Exemple d'ús
node_inicial = "A"
node_final = "I"

final, cost, avaluats = ucs(graf, node_inicial, node_final)
print("Cost:", cost)
print("Avaluats:", avaluats)

Coste: 16
Estados evaluados: 10


Para reconstruir la ruta, modificaremos UCS para que devuelva el camino completo y no solo el coste.

In [10]:
def ucs_ruta(graf, inici, desti):
    # Inicialitzem la llista de nodes visitats
    visitats = set()
    # Inicialitzem la llista de nodes per visitar
    frontera = queue.PriorityQueue()
    # Inicialitzem la frontera amb el node inicial, cost = 0 i camí = [node_inicial]
    frontera.put((0, [node_inicial], [node_inicial]))
    # Per poder comparar els estats avaluats incialitzem avaluats a 0
    avaluats = 0

    # Mentre la frontera no estigui buida
    while not frontera.empty():
        # Treiem el node de la frontera
        cost, node, cami = frontera.get()
        # Incrementem el nombre d'estats avaluats
        avaluats += 1

        # Si el node no ha estat visitat
        if node not in visitats:
            # Marquem el node com a visitat
            visitats.add(node)

            # Si el node és el node final, retornem el camí
            if node == desti:
                return cami, cost, avaluats, cami

            # Afegim els veïns del node a la frontera, cost = ucs(graf, node_inicial, node_final)
            for vei, cost_vei in graf[node].items():
                if vei not in visitats:
                    frontera.put((cost + cost_vei, vei, cami + [vei]))

    # Si no hi ha camí, retornem None
    return None
cost
# Exemple d'ús
node_inicial = "A"
node_final = "I"

final, cost, avaluats, cami = ucs_ruta(graf, node_inicial, node_final)


TypeError: unhashable type: 'list'